# Stronger Model: Random Forest with High-Risk Threshold Tuning

This notebook trains a stronger model for the Early Warning System using the feature-engineered ML-ready data.

The main business goal is not simply overall accuracy. For an Early Warning System, the key question is:

> Can we catch as many truly high-risk product-months as possible without creating too many false high-risk alerts?

Therefore, this notebook focuses on:

- **High-Risk Recall**: how many actual high-risk cases the model catches.
- **False Alarm Rate**: among non-high-risk cases, how many are incorrectly flagged as high risk.
- **High-Risk Precision**: among high-risk alerts, how many are truly high risk.
- **Macro F1**: balanced quality across all classes.

## Modeling Strategy

This notebook uses the feature-engineered ML-ready data and trains a stronger tree-based model.

The workflow is:

1. Train a Random Forest on the chronological training period.
2. Evaluate default multiclass predictions.
3. Tune the High-Risk probability threshold on the validation period.
4. Apply the selected threshold once to the test period.
5. Compare High-Risk Recall, Precision, False Alarm Rate, and Macro F1.

This keeps test evaluation honest while aligning the model with the Early Warning business objective.


In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
TARGET_COL = "next_month_risk_label"
RISK_LABELS = [0, 1, 2]
LABEL_NAMES = {0: "Low Risk", 1: "Medium Risk", 2: "High Risk"}

INPUT_DIR = Path("../data/feature_engineering")
OUTPUT_DIR = Path("../data/ml_stronger")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load feature-engineered train/validation/test files
def load_ml_ready_data(input_dir=INPUT_DIR):
    paths = {
        "train": input_dir / "train_feature_engineered_ml_ready.csv",
        "valid": input_dir / "valid_feature_engineered_ml_ready.csv",
        "test": input_dir / "test_feature_engineered_ml_ready.csv"}
    missing = [str(path) for path in paths.values() if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing ML-ready input files:\n" + "\n".join(missing))
    return {name: pd.read_csv(path) for name, path in paths.items()}


def split_xy(df):
    X = df.drop(columns=[TARGET_COL]).copy()
    y = df[TARGET_COL].astype(int).to_numpy()
    return X, y

In [ ]:
def confusion_matrix_np(y_true, y_pred, labels=RISK_LABELS):
    matrix = np.zeros((len(labels), len(labels)), dtype=int)
    label_to_idx = {label: idx for idx, label in enumerate(labels)}
    for true, pred in zip(y_true, y_pred):
        matrix[label_to_idx[int(true)], label_to_idx[int(pred)]] += 1
    return matrix


def classification_report_np(y_true, y_pred, labels=RISK_LABELS):
    cm = confusion_matrix_np(y_true, y_pred, labels)
    rows = []
    for i, label in enumerate(labels):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        support = cm[i, :].sum()
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append({
            "label": int(label),
            "label_name": LABEL_NAMES[label],
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "support": int(support)})
    report = pd.DataFrame(rows)
    accuracy = np.trace(cm) / cm.sum()
    macro_f1 = report["f1_score"].mean()
    weighted_f1 = np.average(report["f1_score"], weights=report["support"])
    high = high_risk_metrics(y_true, y_pred)
    summary = {
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        **high}
    return report, cm, summary


# Business-focused alert metrics for class 2
def high_risk_metrics(y_true, y_pred):
    y_true_high = y_true == 2
    y_pred_high = y_pred == 2
    tp = int(np.sum(y_true_high & y_pred_high))
    fp = int(np.sum(~y_true_high & y_pred_high))
    fn = int(np.sum(y_true_high & ~y_pred_high))
    tn = int(np.sum(~y_true_high & ~y_pred_high))

    recall = tp / (tp + fn) if (tp + fn) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    false_alarm_rate = fp / (fp + tn) if (fp + tn) else 0.0
    alert_rate = (tp + fp) / len(y_true)

    return {
        "high_risk_recall": float(recall),
        "high_risk_precision": float(precision),
        "false_alarm_rate": float(false_alarm_rate),
        "high_risk_alert_rate": float(alert_rate),
        "high_risk_tp": tp,
        "high_risk_fp": fp,
        "high_risk_fn": fn,
        "high_risk_tn": tn}

In [ ]:
# Small CART-like classification tree for multiclass Random Forest
class RandomTree:
    def __init__(
        self,
        max_depth=8,
        min_samples_split=160,
        min_samples_leaf=70,
        max_features=12,
        n_thresholds=8,
        class_weight=None,
        random_state=42,
    ):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.n_thresholds = n_thresholds
        self.class_weight = class_weight or {0: 1.0, 1: 1.0, 2: 1.0}
        self.rng = np.random.default_rng(random_state)
        self.n_classes_ = 3
        self.feature_importances_ = None

    def fit(self, X, y):
        self.n_features_ = X.shape[1]
        self.feature_importances_ = np.zeros(self.n_features_, dtype=float)
        self.tree_ = self._build(X, y, depth=0)
        total = self.feature_importances_.sum()
        if total > 0:
            self.feature_importances_ = self.feature_importances_ / total
        return self

    def _weighted_counts(self, y):
        counts = np.zeros(self.n_classes_, dtype=float)
        for cls in range(self.n_classes_):
            counts[cls] = np.sum(y == cls) * self.class_weight.get(cls, 1.0)
        return counts

    def _proba(self, y):
        counts = self._weighted_counts(y) + 1e-6
        return counts / counts.sum()

    def _gini_from_counts(self, counts):
        total = counts.sum()
        if total <= 0:
            return 0.0
        p = counts / total
        return 1.0 - np.sum(p ** 2)

    def _node_gini(self, y):
        return self._gini_from_counts(self._weighted_counts(y))

    def _candidate_thresholds(self, x):
        x = x[np.isfinite(x)]
        if len(x) < 2:
            return np.array([])
        qs = np.linspace(0.10, 0.90, self.n_thresholds)
        thresholds = np.unique(np.quantile(x, qs))
        return thresholds

    def _best_split(self, X, y):
        parent_gini = self._node_gini(y)
        best_gain = 0.0
        best_feature = None
        best_threshold = None
        best_left = None

        n_features_to_try = min(self.max_features, X.shape[1])
        feature_ids = self.rng.choice(X.shape[1], size=n_features_to_try, replace=False)

        for feature in feature_ids:
            x = X[:, feature]
            thresholds = self._candidate_thresholds(x)
            if len(thresholds) == 0:
                continue

            for threshold in thresholds:
                left = x <= threshold
                n_left = int(left.sum())
                n_right = len(y) - n_left
                if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
                    continue

                y_left = y[left]
                y_right = y[~left]
                left_counts = self._weighted_counts(y_left)
                right_counts = self._weighted_counts(y_right)
                total_weight = left_counts.sum() + right_counts.sum()
                weighted_child_gini = (
                    left_counts.sum() / total_weight * self._gini_from_counts(left_counts)
                    + right_counts.sum() / total_weight * self._gini_from_counts(right_counts)
                )
                gain = parent_gini - weighted_child_gini

                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold
                    best_left = left

        return best_feature, best_threshold, best_left, best_gain

    def _build(self, X, y, depth):
        proba = self._proba(y)
        node = {
            "proba": proba,
            "class": int(np.argmax(proba)),
            "n_samples": int(len(y)),
        }

        if (
            depth >= self.max_depth
            or len(y) < self.min_samples_split
            or len(np.unique(y)) == 1
        ):
            return node

        feature, threshold, left, gain = self._best_split(X, y)
        if feature is None or gain <= 1e-10:
            return node

        self.feature_importances_[feature] += gain * len(y)
        node["feature"] = int(feature)
        node["threshold"] = float(threshold)
        node["left"] = self._build(X[left], y[left], depth + 1)
        node["right"] = self._build(X[~left], y[~left], depth + 1)
        return node

    def _predict_one(self, row, node):
        while "feature" in node:
            if row[node["feature"]] <= node["threshold"]:
                node = node["left"]
            else:
                node = node["right"]
        return node["proba"]

    def predict_proba(self, X):
        return np.vstack([self._predict_one(row, self.tree_) for row in X])

# Compact Random Forest classifier using the RandomTree above
class RandomForestNumpy:

    def __init__(
        self,
        n_estimators=35,
        max_depth=8,
        min_samples_split=160,
        min_samples_leaf=70,
        max_features="sqrt",
        n_thresholds=8,
        row_subsample=0.72,
        class_weight=None,
        random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.n_thresholds = n_thresholds
        self.row_subsample = row_subsample
        self.class_weight = class_weight
        self.random_state = random_state

    def fit(self, X, y, feature_names):
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=int)
        self.feature_names_ = np.array(feature_names)
        self.n_features_ = X.shape[1]

        if self.max_features == "sqrt":
            max_features = max(2, int(np.sqrt(self.n_features_)))
        else:
            max_features = int(self.max_features)

        rng = np.random.default_rng(self.random_state)
        self.trees_ = []
        self.feature_importances_ = np.zeros(self.n_features_, dtype=float)

        n_rows = X.shape[0]
        sample_size = int(n_rows * self.row_subsample)

        for i in range(self.n_estimators):
            idx = rng.choice(n_rows, size=sample_size, replace=True)
            tree = RandomTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf,
                max_features=max_features,
                n_thresholds=self.n_thresholds,
                class_weight=self.class_weight,
                random_state=self.random_state + i + 1,
            )
            tree.fit(X[idx], y[idx])
            self.trees_.append(tree)
            self.feature_importances_ += tree.feature_importances_

        total = self.feature_importances_.sum()
        if total > 0:
            self.feature_importances_ = self.feature_importances_ / total
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=np.float32)
        probs = np.zeros((X.shape[0], 3), dtype=float)
        for tree in self.trees_:
            probs += tree.predict_proba(X)
        return probs / len(self.trees_)

    def predict(self, X): return np.argmax(self.predict_proba(X), axis=1)

# Prioritize high-risk alerts above a selected probability threshold
def predict_with_high_risk_threshold(proba, threshold):
    pred = np.argmax(proba, axis=1)
    non_high_best = np.argmax(proba[:, :2], axis=1)
    pred = np.where(proba[:, 2] >= threshold, 2, non_high_best)
    return pred.astype(int)

# Find a threshold with strong high-risk recall and controlled false alarms
def tune_high_risk_threshold(y_true, proba, thresholds=None, min_recall=0.70):
    if thresholds is None:
        thresholds = np.round(np.arange(0.20, 0.81, 0.02), 2)

    rows = []
    for threshold in thresholds:
        pred = predict_with_high_risk_threshold(proba, threshold)
        report, cm, summary = classification_report_np(y_true, pred)
        rows.append({"threshold": float(threshold), **summary})

    table = pd.DataFrame(rows)

    candidates = table[table["high_risk_recall"] >= min_recall].copy()
    if len(candidates):
        best = candidates.sort_values(
            ["false_alarm_rate", "high_risk_precision", "macro_f1"],
            ascending=[True, False, False],
        ).iloc[0]
    else:
        best = table.sort_values(
            ["high_risk_recall", "macro_f1", "false_alarm_rate"],
            ascending=[False, False, True],
        ).iloc[0]

    return table, float(best["threshold"])

In [ ]:
def feature_importance_table(model):
    return (
        pd.DataFrame({
            "feature": model.feature_names_,
            "importance": model.feature_importances_})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True))

def make_recommendations(summary_default, summary_tuned):
    recs = []

    if summary_tuned["high_risk_recall"] > summary_default["high_risk_recall"]:
        recs.append({
            "area": "Threshold strategy",
            "recommendation": "Use the tuned high-risk threshold when the business priority is catching more high-risk cases."})
    else:
        recs.append({
            "area": "Threshold strategy",
            "recommendation": "The default argmax threshold is already competitive; keep threshold tuning as a monitored option."})

    if summary_tuned["false_alarm_rate"] > 0.35:
        recs.append({
            "area": "False alarms",
            "recommendation": "False alarm rate is high. Tighten the high-risk threshold or add a second-stage review score before alerts go to business users."})
    else:
        recs.append({
            "area": "False alarms",
            "recommendation": "False alarm rate is acceptable for a first warning model, but should be validated with business capacity for alert handling."})

    if summary_tuned["high_risk_recall"] < 0.70:
        recs.append({
            "area": "High-risk recall",
            "recommendation": "High-risk recall is below the desired 70% level. Add more leading indicators or use XGBoost/LightGBM with class weighting next."})
    else:
        recs.append({
            "area": "High-risk recall",
            "recommendation": "High-risk recall reaches the initial target. Next optimize precision and alert volume."})

    recs.append({
        "area": "Production next step",
        "recommendation": "Replace the NumPy forest with scikit-learn RandomForest/GradientBoosting or XGBoost/LightGBM when libraries are available, then compare metrics on the same time-based split."})

    return pd.DataFrame(recs)



In [ ]:
def run_stronger_model_pipeline():
    print("Loading feature-engineered ML-ready files...")
    data = load_ml_ready_data()
    X_train_df, y_train = split_xy(data["train"])
    X_valid_df, y_valid = split_xy(data["valid"])
    X_test_df, y_test = split_xy(data["test"])

    print("Input shapes")
    print("-" * 80)
    print(f"Train: {X_train_df.shape}")
    print(f"Valid: {X_valid_df.shape}")
    print(f"Test:  {X_test_df.shape}")

    # Train on a deterministic sample for speed while preserving class distribution
    train_for_fit = data["train"].groupby(TARGET_COL, group_keys=False).sample(
        frac=0.70,
        random_state=RANDOM_SEED)
    X_fit_df, y_fit = split_xy(train_for_fit)

    class_counts = pd.Series(y_fit).value_counts().to_dict()
    max_count = max(class_counts.values())
    class_weight = {int(cls): float(max_count / count) for cls, count in class_counts.items()}
    class_weight[2] = class_weight.get(2, 1.0) * 1.35

    print("\nTraining NumPy Random Forest...")
    print("-" * 80)
    print(f"Fit sample shape: {X_fit_df.shape}")
    print(f"Class weights: {class_weight}")

    model = RandomForestNumpy(
        n_estimators=35,
        max_depth=8,
        min_samples_split=180,
        min_samples_leaf=75,
        max_features="sqrt",
        n_thresholds=8,
        row_subsample=0.74,
        class_weight=class_weight,
        random_state=RANDOM_SEED)
    model.fit(X_fit_df.to_numpy(dtype=np.float32), y_fit, X_fit_df.columns)

    valid_proba = model.predict_proba(X_valid_df.to_numpy(dtype=np.float32))
    test_proba = model.predict_proba(X_test_df.to_numpy(dtype=np.float32))

    valid_default_pred = np.argmax(valid_proba, axis=1)
    test_default_pred = np.argmax(test_proba, axis=1)

    valid_default_report, valid_default_cm, valid_default_summary = classification_report_np(y_valid, valid_default_pred)
    test_default_report, test_default_cm, test_default_summary = classification_report_np(y_test, test_default_pred)

    threshold_table, best_threshold = tune_high_risk_threshold(y_valid, valid_proba, min_recall=0.70)
    valid_tuned_pred = predict_with_high_risk_threshold(valid_proba, best_threshold)
    test_tuned_pred = predict_with_high_risk_threshold(test_proba, best_threshold)

    valid_tuned_report, valid_tuned_cm, valid_tuned_summary = classification_report_np(y_valid, valid_tuned_pred)
    test_tuned_report, test_tuned_cm, test_tuned_summary = classification_report_np(y_test, test_tuned_pred)

    summary = pd.DataFrame([
        {"model": "numpy_random_forest", "split": "valid", "decision_rule": "default_argmax", "threshold": np.nan, **valid_default_summary},
        {"model": "numpy_random_forest", "split": "test", "decision_rule": "default_argmax", "threshold": np.nan, **test_default_summary},
        {"model": "numpy_random_forest", "split": "valid", "decision_rule": "high_risk_threshold", "threshold": best_threshold, **valid_tuned_summary},
        {"model": "numpy_random_forest", "split": "test", "decision_rule": "high_risk_threshold", "threshold": best_threshold, **test_tuned_summary}])

    importance = feature_importance_table(model)
    recommendations = make_recommendations(test_default_summary, test_tuned_summary)

    summary.to_csv(OUTPUT_DIR / "stronger_model_results.csv", index=False)
    threshold_table.to_csv(OUTPUT_DIR / "high_risk_threshold_tuning_valid.csv", index=False)
    importance.to_csv(OUTPUT_DIR / "random_forest_feature_importance.csv", index=False)
    recommendations.to_csv(OUTPUT_DIR / "stronger_model_recommendations.csv", index=False)

    pd.DataFrame(valid_default_cm, index=[f"actual_{LABEL_NAMES[i]}" for i in RISK_LABELS], columns=[f"pred_{LABEL_NAMES[i]}" for i in RISK_LABELS]).to_csv(OUTPUT_DIR / "valid_default_confusion_matrix.csv")
    pd.DataFrame(test_default_cm, index=[f"actual_{LABEL_NAMES[i]}" for i in RISK_LABELS], columns=[f"pred_{LABEL_NAMES[i]}" for i in RISK_LABELS]).to_csv(OUTPUT_DIR / "test_default_confusion_matrix.csv")
    pd.DataFrame(valid_tuned_cm, index=[f"actual_{LABEL_NAMES[i]}" for i in RISK_LABELS], columns=[f"pred_{LABEL_NAMES[i]}" for i in RISK_LABELS]).to_csv(OUTPUT_DIR / "valid_tuned_confusion_matrix.csv")
    pd.DataFrame(test_tuned_cm, index=[f"actual_{LABEL_NAMES[i]}" for i in RISK_LABELS], columns=[f"pred_{LABEL_NAMES[i]}" for i in RISK_LABELS]).to_csv(OUTPUT_DIR / "test_tuned_confusion_matrix.csv")

    valid_default_report.to_csv(OUTPUT_DIR / "valid_default_classification_report.csv", index=False)
    test_default_report.to_csv(OUTPUT_DIR / "test_default_classification_report.csv", index=False)
    valid_tuned_report.to_csv(OUTPUT_DIR / "valid_tuned_classification_report.csv", index=False)
    test_tuned_report.to_csv(OUTPUT_DIR / "test_tuned_classification_report.csv", index=False)

    proba_sample = pd.DataFrame({
        "valid_actual": y_valid,
        "valid_pred_default": valid_default_pred,
        "valid_pred_tuned": valid_tuned_pred,
        "proba_low": valid_proba[:, 0],
        "proba_medium": valid_proba[:, 1],
        "proba_high": valid_proba[:, 2]})
    proba_sample.to_csv(OUTPUT_DIR / "valid_prediction_probabilities.csv", index=False)

    report = {
        "model": "NumPy Random Forest",
        "best_high_risk_threshold": best_threshold,
        "summary": summary.to_dict(orient="records"),
        "top_20_features": importance.head(20).to_dict(orient="records"),
        "recommendations": recommendations.to_dict(orient="records")}
    with open(OUTPUT_DIR / "stronger_model_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, default=str)

    print("\nStronger model results")
    print("-" * 80)
    print(summary.round(4).to_string(index=False))

    print("\nBest high-risk threshold from validation")
    print("-" * 80)
    print(best_threshold)

    print("\nTest confusion matrix with tuned high-risk threshold")
    print("-" * 80)
    print(pd.DataFrame(test_tuned_cm, index=[f"actual_{LABEL_NAMES[i]}" for i in RISK_LABELS], columns=[f"pred_{LABEL_NAMES[i]}" for i in RISK_LABELS]).to_string())

    print("\nTest classification report with tuned high-risk threshold")
    print("-" * 80)
    print(test_tuned_report.round(4).to_string(index=False))

    print("\nTop 20 Random Forest feature importances")
    print("-" * 80)
    print(importance.head(20).round(5).to_string(index=False))

    print("\nRecommendations")
    print("-" * 80)
    print(recommendations.to_string(index=False))

    print("\nOutput files saved to:", OUTPUT_DIR.resolve())

    return {
        "summary": summary,
        "threshold_table": threshold_table,
        "feature_importance": importance,
        "recommendations": recommendations}

In [7]:
artifacts = run_stronger_model_pipeline()

Loading feature-engineered ML-ready files...
Input shapes
--------------------------------------------------------------------------------
Train: (62520, 164)
Valid: (18000, 164)
Test:  (16000, 164)

Training NumPy Random Forest...
--------------------------------------------------------------------------------
Fit sample shape: (43764, 164)
Class weights: {1: 1.0, 2: 2.053455176093917, 0: 3.821045576407507}

Stronger model results
--------------------------------------------------------------------------------
              model split       decision_rule  threshold  accuracy  macro_f1  weighted_f1  high_risk_recall  high_risk_precision  false_alarm_rate  high_risk_alert_rate  high_risk_tp  high_risk_fp  high_risk_fn  high_risk_tn
numpy_random_forest valid      default_argmax        NaN    0.6961    0.7029       0.6941            0.9423               0.5574            0.4056                0.5943          5963          4734           365          6938
numpy_random_forest  test      de